# Lab 01 — Part G–I: Search, evaluation and error analysis

Notebook thực hiện các yêu cầu ở mục 10, 11 và 12: xây TF-IDF search index, trả về Top-5 results, đo Precision@5/Recall@5/MRR trên tập nhãn nhỏ và xuất report error analysis.

In [ ]:
import sys
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
CANDIDATE_EXERCISE_DIRECTORIES = (
    WORKING_DIRECTORY,
    WORKING_DIRECTORY / "Chapter10",
    WORKING_DIRECTORY / "lab1" / "Chapter10",
)
EXERCISE_DIRECTORY = next(
    path
    for path in CANDIDATE_EXERCISE_DIRECTORIES
    if (path / "run_search_evaluation.py").is_file()
)
if str(EXERCISE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(EXERCISE_DIRECTORY))

from run_search_evaluation import (
    DATASET_PATH,
    PIPELINE_NAME,
    QUERY_LABELS,
    REPORT_PATH,
    RESULTS_PATH,
    TOP_DOCUMENT_COUNT,
    PipelineResult,
    calculate_metrics,
    evaluate_queries,
    fit_search_index,
    load_documents,
    rank_documents,
    write_report,
    write_results,
)


In [ ]:
# 10.1 — Bài toán: đọc 30K documents làm nguồn cho document search.
documents = load_documents(DATASET_PATH)

print(f"Documents indexed: {len(documents):,}")
print(f"Selected pipeline: {PIPELINE_NAME}")


In [ ]:
# 10.2 — Pipeline: documents → TF-IDF index → query vector → cosine similarity → ranking.
vectorizer, document_matrix = fit_search_index(documents)
search_index = PipelineResult(PIPELINE_NAME, vectorizer, document_matrix)

print(f"TF-IDF matrix shape: {document_matrix.shape}")
print(f"Non-zero values: {document_matrix.nnz:,}")


In [ ]:
# 10.3 — Query examples: chạy các query có relevance labels.
search_rankings = {
    query: rank_documents(search_index, query, TOP_DOCUMENT_COUNT)
    for query in QUERY_LABELS
}
ranking_lines = tuple(
    f"{query} | rank {rank} | document {document_id} | similarity={similarity:.6f} | "
    f"{documents[document_id].replace(chr(10), ' ')[:120]}"
    for query, ranking in search_rankings.items()
    for rank, (document_id, similarity) in enumerate(ranking, start=1)
)

print("\n".join(ranking_lines))


In [ ]:
# 10.4 — Output: lưu Top-5 document ID, similarity và preview cho từng query.
retrieval_rows, metric_rows = evaluate_queries(documents, vectorizer, document_matrix)
write_results(retrieval_rows, metric_rows)

print(f"Saved {len(retrieval_rows)} retrieval rows and {len(metric_rows)} metric rows to {RESULTS_PATH.name}")


In [ ]:
# 11.1 — Evaluation set: seven queries with manually recorded relevance labels.
label_lines = tuple(
    f"{query}: relevant document IDs = {sorted(relevant_ids)}"
    for query, relevant_ids in QUERY_LABELS.items()
)

print("\n".join(label_lines))


In [ ]:
# 11.2–11.4 — Precision@5, Recall@5 và MRR cho từng query và trung bình.
metric_values = tuple(
    calculate_metrics(
        [document_id for document_id, _ in search_rankings[query]],
        relevant_ids,
    )
    for query, relevant_ids in QUERY_LABELS.items()
)
metric_lines = tuple(
    f"{query}: P@5={precision:.3f}; Recall@5={recall:.3f}; RR={reciprocal_rank:.3f}"
    for query, (precision, recall, reciprocal_rank) in zip(QUERY_LABELS, metric_values)
)
mean_precision = sum(value[0] for value in metric_values) / len(metric_values)
mean_recall = sum(value[1] for value in metric_values) / len(metric_values)
mean_reciprocal_rank = sum(value[2] for value in metric_values) / len(metric_values)

print("\n".join(metric_lines))
print(f"\nMean P@5={mean_precision:.3f}; Mean Recall@5={mean_recall:.3f}; MRR={mean_reciprocal_rank:.3f}")


In [ ]:
# 12 — Error analysis: xuất report có hai good cases, hai weak cases và failure quan trọng nhất.
from IPython.display import Markdown, display

write_report(metric_rows)
display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))
